# Stage 3 — Preference Tuning with DPO
### Load the Stage-2 (SFT) model from the Hub → align with chosen/rejected pairs → push to the Hub

This is **notebook 3 of 3**. It downloads your Stage-2 instruction model and
fine-tunes it with **Direct Preference Optimization (DPO)** on medical
`chosen` vs `rejected` answer pairs, then pushes the aligned model.

**Concept — preference learning vs. SFT.** SFT learns from *one gold answer*
per prompt. Preference learning instead learns from *comparisons*: for the
same prompt it sees a **chosen** (better) and a **rejected** (worse) answer,
and learns to make the chosen one more likely than the rejected one.

**Concept — what DPO does.** Classic RLHF trains a separate reward model and
then runs reinforcement learning. **DPO skips the reward model**: it directly
adjusts the policy so the log-probability of *chosen* rises relative to
*rejected*, while a frozen **reference** copy of the model (here, the Stage-2
weights with the adapter disabled) keeps it from drifting too far. The
strength of that anchor is the hyperparameter **`beta`**.

```text
Stage 2 model (Hub)  →  + preference data (chosen/rejected)  →  push Stage 3 model (Hub)
```

In [ ]:
# ============================================================
# Step 1. Install libraries (note: trl is added for DPO)
# ============================================================
!pip install -q -U datasets transformers accelerate peft bitsandbytes sentencepiece huggingface_hub trl

In [ ]:
# ============================================================
# Step 2. Imports
# ============================================================
import os, gc, json
from dataclasses import dataclass, asdict

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import DPOTrainer, DPOConfig

## Hugging Face login

We push every stage's model to the Hub, and each later notebook **pulls the
previous stage's model from the Hub**. So you must be logged in.

Two ways to provide your token (create one at
https://huggingface.co/settings/tokens with *write* access):

- **Easiest in Colab:** open the key icon on the left, add a secret named
  `HF_TOKEN`, then run the cell below — it reads the secret automatically.
- **Or** just run `login()` and paste the token when prompted.

In [ ]:
# ============================================================
# Hugging Face login
# ============================================================
from huggingface_hub import login, whoami

try:
    # Colab secret named HF_TOKEN (recommended).
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    # Fallback: paste the token when prompted.
    login()

HF_USERNAME = whoami()["name"]
print("Logged in as:", HF_USERNAME)

## Step 3 — Configuration

Point `stage2_repo_name` at your Stage-2 repo. The preference dataset
`empirischtech/med-qa-orpo-dpo` is built from medical QA sources and is
already shaped as `question` / `chosen` / `rejected`.

In [ ]:
# ============================================================
# Step 3. Configuration
# ============================================================
@dataclass
class Config:
    # Stage-2 output is THIS stage's base model.
    stage2_repo_name: str = "med-tinyllama-stage2-sft"

    # Preference (DPO) dataset.
    dataset_name: str = "empirischtech/med-qa-orpo-dpo"
    n_samples: int = 1500               # subsample for a fast Colab demo

    # Hub repo NAME for this stage's output.
    stage3_repo_name: str = "med-tinyllama-stage3-dpo"
    private_repo: bool = True

    output_dir: str = "/content/stage3_output"
    adapter_dir: str = "/content/stage3_adapter"
    merged_dir: str = "/content/stage3_merged"

    test_size: float = 0.1
    seed: int = 42

    # DPO hyperparameters.
    beta: float = 0.1                   # how strongly to stay near the reference model
    max_length: int = 512               # full sequence length cap
    max_prompt_length: int = 256        # prompt length cap

    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    num_train_epochs: float = 1.0
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 5e-5         # DPO uses a small learning rate
    warmup_steps: int = 5
    logging_steps: int = 5
    max_steps: int = -1


config = Config()
for d in (config.output_dir, config.adapter_dir, config.merged_dir):
    os.makedirs(d, exist_ok=True)

STAGE2_REPO = f"{HF_USERNAME}/{config.stage2_repo_name}"   # base for this stage
STAGE3_REPO = f"{HF_USERNAME}/{config.stage3_repo_name}"   # output of this stage
print("Base model (from Stage 2):", STAGE2_REPO)
print("Will push Stage 3 to:     ", STAGE3_REPO)

## Step 4 — Load and shape the preference data

`trl`'s `DPOTrainer` expects three text columns: **`prompt`**, **`chosen`**,
**`rejected`**. The loader below auto-detects the prompt column (it may be
called `question` or `prompt`) and wraps it in the same instruction template
the model learned in Stage 2, so the preference signal lines up with how the
model was taught to answer.

In [ ]:
# ============================================================
# Step 4. Load preference data, subsample, map to prompt/chosen/rejected
# ============================================================
pref = load_dataset(config.dataset_name, split="train")
pref = pref.shuffle(seed=config.seed).select(range(min(config.n_samples, len(pref))))
print("Columns:", pref.column_names)

# Find the prompt-like column (question / prompt / instruction).
cols = pref.column_names
prompt_col = next((x for x in ["prompt", "question", "instruction"] if x in cols), None)
if prompt_col is None or "chosen" not in cols or "rejected" not in cols:
    raise ValueError(f"Expected a prompt column plus chosen/rejected. Got: {cols}")
print("Using prompt column:", prompt_col)


def to_dpo_format(record):
    instruction = str(record[prompt_col]).strip()
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"   # match Stage-2 template
    return {"prompt": prompt,
            "chosen": str(record["chosen"]).strip(),
            "rejected": str(record["rejected"]).strip()}


pref = pref.map(to_dpo_format, remove_columns=cols)
pref = pref.filter(lambda r: len(r["chosen"]) > 0 and len(r["rejected"]) > 0)
pref = pref.train_test_split(test_size=config.test_size, seed=config.seed)
pref["validation"] = pref.pop("test")
print(pref)
print("\nExample prompt:\n", pref["train"][0]["prompt"])
print("chosen   :", pref["train"][0]["chosen"][:160])
print("rejected :", pref["train"][0]["rejected"][:160])

## Step 5 — Load the Stage-2 model from the Hub (the policy)

We load the **merged Stage-2 model** in 4-bit. We pass a LoRA config to the
`DPOTrainer` (`peft_config`), so it trains a small adapter as the *policy*
and automatically uses the same base **with the adapter disabled** as the
frozen *reference* — no separate reference model needed.

In [ ]:
# ============================================================
# Step 5. Tokenizer + Stage-2 policy model (4-bit) + LoRA config for DPO
# ============================================================
use_cuda = torch.cuda.is_available()
print("CUDA:", use_cuda)
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(STAGE2_REPO, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if use_cuda:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    policy_model = AutoModelForCausalLM.from_pretrained(
        STAGE2_REPO, quantization_config=bnb, device_map="auto", trust_remote_code=True)
    policy_model = prepare_model_for_kbit_training(policy_model)
else:
    policy_model = AutoModelForCausalLM.from_pretrained(
        STAGE2_REPO, torch_dtype=torch.float32, trust_remote_code=True)
policy_model.config.use_cache = False

dpo_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=config.lora_r, lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
print("Policy model loaded.")

## Step 6 — Train with DPO

> **trl version note.** This uses the modern API: hyperparameters live in
> `DPOConfig`, and the tokenizer is passed as `processing_class`. On older
> `trl` (< 0.12) use `tokenizer=tokenizer` instead of `processing_class=` and
> move `beta` / `max_length` / `max_prompt_length` into `DPOTrainer(...)`.

In [ ]:
# ============================================================
# Step 6. DPOConfig + DPOTrainer + train
# ============================================================
dpo_args = DPOConfig(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=config.warmup_steps,
    logging_steps=config.logging_steps, logging_first_step=True,
    save_strategy="no",
    fp16=use_cuda, bf16=False, report_to="none",
    beta=config.beta,
    max_length=config.max_length,
    max_prompt_length=config.max_prompt_length,
)

dpo_trainer = DPOTrainer(
    model=policy_model,
    ref_model=None,                     # PEFT: reference = base with adapter disabled
    args=dpo_args,
    train_dataset=pref["train"],
    eval_dataset=pref["validation"],
    processing_class=tokenizer,         # older trl: use tokenizer=tokenizer
    peft_config=dpo_lora_config,
)
print("Training (DPO)...")
dpo_trainer.train()
print("Done.")

## Step 7 — Merge and push to the Hub

In [ ]:
# ============================================================
# Step 7. Save adapter, merge onto Stage-2 base, push to Hub
# ============================================================
dpo_trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

del dpo_trainer, policy_model
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

base_fp = AutoModelForCausalLM.from_pretrained(
    STAGE2_REPO,
    torch_dtype=torch.float16 if use_cuda else torch.float32,
    device_map="auto" if use_cuda else None, trust_remote_code=True)
merged = PeftModel.from_pretrained(base_fp, config.adapter_dir).merge_and_unload()
merged.save_pretrained(config.merged_dir)
tokenizer.save_pretrained(config.merged_dir)

merged.push_to_hub(STAGE3_REPO, private=config.private_repo)
tokenizer.push_to_hub(STAGE3_REPO, private=config.private_repo)
print(f"Stage 3 (DPO) model pushed to: https://huggingface.co/{STAGE3_REPO}")

## Step 8 — Inference

Same instruction template as Stage 2 — now the answers should reflect the
preference signal (clearer, safer, more on-point).

In [ ]:
# ============================================================
# Step 8. Inference with the DPO-aligned model
# ============================================================
merged.eval()
device = merged.device


def ask(instruction, max_new_tokens=150):
    prompt = f"### Instruction:\n{instruction.strip()}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = merged.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                              temperature=0.7, top_p=0.9, repetition_penalty=1.1,
                              pad_token_id=tokenizer.eos_token_id,
                              eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)


for q in ["What is the primary mechanism of action of metformin?",
          "A patient asks whether they can stop their statin once cholesterol is normal. How should this be addressed?"]:
    print("=" * 90)
    print("Q:", q)
    print(ask(q))

## Done — full pipeline complete

You have trained one model through all three stages, each pushed to the Hub:

1. **Stage 1** — self-supervised continued pretraining (learned the language)
2. **Stage 2** — supervised instruction tuning (learned to answer)
3. **Stage 3** — DPO preference tuning (learned to prefer better answers)

The final model lives at `STAGE3_REPO`.

> **Reminder.** These models are for learning only — medical outputs from a
> 1.1B demo model are not clinical advice.